# 4. Measurements, peak frequency and Wang's change points

Acquisition chain (FM-09, FM-22): resample to 2 kHz, Gaussian noise per electrode, 30-500 Hz band-pass
for bipoles and omnipoles. Peak frequency (FM-10): Morlet wavelet approximation of EnSite OT Near
Field, two co-primary definitions (FM-27); bipolar/omnipolar PP within +-30 ms of the peak-frequency
time (FM-11). Amplitudes stay in model units; Wang's 1.44 mV is shown beside them, never applied (FM-25).
The export writes Wang's EnSite columns; her Rmd then runs unchanged except six one-slab functions
(R/wang_replication.R, FM-12).

In [ ]:
import os, sys
from pathlib import Path
REPOSITORY = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPOSITORY))

# ---- settings: edit here, or set EFOV_CONFIG / EFOV_OUT / EFOV_NP ----
CONFIG = os.environ.get("EFOV_CONFIG", str(REPOSITORY / "config" / "model.json"))
OUT    = os.environ.get("EFOV_OUT",    str(REPOSITORY / "runs" / "v4_reference"))
NP     = int(os.environ.get("EFOV_NP", "24"))      # MPI processes for openCARP
# openCARP's own MPI launcher (efov-runner image); needed when NP > 1
os.environ.setdefault("MPIEXEC_BIN", "/usr/local/lib/opencarp/lib/petsc/bin/mpiexec")

import json
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display
from efov.config import loadConfig
from efov.pipeline import Run

config = loadConfig(CONFIG)
run = Run(config, OUT, processCount=NP)
print(f"{config['modelName']} -> {run.out}")
run.status()

In [ ]:
measures = run.runStage("measurements")
measures["amplitudes"], measures["checks"]

In [ ]:
points = pd.read_csv(run.out / "measurements" / "lesionPoints.csv")
points[points.noiseRepeat == 0].groupby("kind")[["ppMv", "pfHz", "pfGlobalHz"]].describe().T

In [ ]:
run.runStage("export")

In [ ]:
changes = run.runStage("changePoints")
print("eligible:", changes["eligibleForInterpretation"], changes["prerequisites"])
pd.read_csv(run.out / "changePoints" / "SuppResults5_change_points_all.csv")

In [ ]:
from efov.report import profileFigure
display(Image(str(profileFigure(run, run.folder("figures"), "pfHz", "peak frequency (Hz)", "03_peak_frequency_profiles.png", log=False))))